# 04 — Hyper-parameter Tests

All bandit hyper-parameter decisions are made here, *before* the full experiments in 05.

1. **Reward function** (§1) — euclidean `r = 1/(1+d)` wins pairwise-accuracy.
2. **Cluster count K** (§2) — K=20 via ε-greedy sweep K ∈ {5, …, 50}.
3. **LinUCB α** (§3) — α=0.5 via N=10, T=50,000 sweep over {0.3, 0.5, 0.8, 1.0, 1.5}.
4. **Contextual TS ν** (§4) — ν=0.3 via sweep over {0.1, 0.3, 0.5, 0.8, 1.0}.
5. **Non-contextual params** (§5) — ε, c, B, γ swept at N=10, T=50,000.
6. **Robustness checks** (§6) — paired t-tests + 95% CIs on every sweep, plus convergence-window sensitivity.

Consumes `outputs/taste_profiles.csv`, `outputs/user_liked_songs.csv`, and `outputs/tempo_params.json` from 03.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Load data
songs = pd.read_csv('outputs/spotify_cleaned.csv')
playlists = pd.read_csv('outputs/user_liked_songs.csv')
profiles = pd.read_csv('outputs/taste_profiles.csv', index_col='pid')

FEATS = ['Danceability', 'Energy', 'Valence', 'Acousticness',
         'Instrumentalness', 'Speechiness', 'Tempo']

# Load tempo params from 03 (avoids magic numbers that drift silently)
import json
with open('outputs/tempo_params.json') as f:
    _tp = json.load(f)
TEMPO_MIN, TEMPO_MAX = _tp['tempo_min'], _tp['tempo_max']
songs['Tempo'] = (songs['Tempo'] - TEMPO_MIN) / (TEMPO_MAX - TEMPO_MIN)


print(f"Songs: {len(songs)}")
print(f"Users (taste profiles): {len(profiles)}")
print(f"Features: {FEATS}")

---
## 1. Reward Function Test

**Goal:** Find which distance metric best separates songs a user actually likes (from their playlist) vs random songs.

**Method:** For 100 random users:
- Take their playlist songs ("liked")
- Sample the same number of random songs not in their playlist ("random")
- Compute each metric for both groups
- A good metric should give liked songs **higher** scores than random songs

**Metrics tested:**
- **Cosine Similarity** — measures angle between vectors (direction only, ignores magnitude)
- **Euclidean Distance** — straight-line distance in feature space (magnitude-sensitive)
- **Manhattan Distance** — sum of absolute differences per feature

All distance metrics are converted to reward using: `reward = 1 / (1 + distance)`

In [ ]:
np.random.seed(42)
sample_pids = profiles.sample(100).index

results = {k: {'liked': [], 'random': []} for k in ['cosine', 'euclidean', 'manhattan']}

for pid in sample_pids:
    user_profile = profiles.loc[pid, FEATS].values

    # Get user's actual liked songs
    user_songs = playlists[playlists['pid'] == pid]
    liked = songs.merge(user_songs, left_on=['Artist', 'Track'],
                        right_on=['artist_name', 'track_name'])
    if len(liked) < 3:
        continue
    liked_features = liked[FEATS].values

    # Get random songs not in playlist
    not_liked_mask = ~songs.set_index(['Artist', 'Track']).index.isin(
        liked.set_index(['Artist', 'Track']).index
    )
    random_songs = songs[not_liked_mask].sample(len(liked), random_state=pid)
    random_features = random_songs[FEATS].values

    for label, feat_array in [('liked', liked_features), ('random', random_features)]:
        # Cosine similarity
        cos = cosine_similarity(feat_array, user_profile.reshape(1, -1)).flatten()
        results['cosine'][label].extend(cos)

        # Euclidean reward
        eucl = 1 / (1 + np.linalg.norm(feat_array - user_profile, axis=1))
        results['euclidean'][label].extend(eucl)

        # Manhattan reward
        manh = 1 / (1 + np.sum(np.abs(feat_array - user_profile), axis=1))
        results['manhattan'][label].extend(manh)

print("Done. Computing results...\n")

In [ ]:
# Compute metrics for each reward function
print("=" * 70)
print("REWARD FUNCTION COMPARISON")
print("=" * 70)
print()
print("A good reward function gives HIGHER scores to liked songs")
print("and LOWER scores to random songs.")
print()

metric_results = {}

for metric, name in [('cosine', 'Cosine Similarity'),
                     ('euclidean', 'Euclidean (1/(1+d))'),
                     ('manhattan', 'Manhattan (1/(1+d))')]:
    liked = np.array(results[metric]['liked'])
    rand = np.array(results[metric]['random'])
    gap = liked.mean() - rand.mean()

    # Pairwise accuracy: how often does a liked song score higher than a random song?
    correct = 0
    total = 5000
    np.random.seed(42)
    for _ in range(total):
        i = np.random.randint(len(liked))
        j = np.random.randint(len(rand))
        if liked[i] > rand[j]:
            correct += 1
    acc = correct / total

    metric_results[metric] = {'liked_avg': liked.mean(), 'random_avg': rand.mean(),
                              'gap': gap, 'accuracy': acc}

    print(f"{name}")
    print(f"  Liked songs avg:    {liked.mean():.4f}")
    print(f"  Random songs avg:   {rand.mean():.4f}")
    print(f"  Gap:                {gap:.4f}")
    print(f"  Pairwise accuracy:  {acc:.1%}")
    print()

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (metric, name) in zip(axes, [('cosine', 'Cosine Similarity'),
                                      ('euclidean', 'Euclidean (1/(1+d))'),
                                      ('manhattan', 'Manhattan (1/(1+d))')]):
    liked = np.array(results[metric]['liked'])
    rand = np.array(results[metric]['random'])

    ax.hist(liked, bins=50, alpha=0.6, label='Liked songs', color='green', density=True)
    ax.hist(rand, bins=50, alpha=0.6, label='Random songs', color='red', density=True)
    ax.axvline(liked.mean(), color='green', linestyle='--', linewidth=2)
    ax.axvline(rand.mean(), color='red', linestyle='--', linewidth=2)
    ax.set_title(f'{name}\nAccuracy: {metric_results[metric]["accuracy"]:.1%}')
    ax.set_xlabel('Reward Score')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Reward Function Comparison: Liked vs Random Songs', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 70)
print("REWARD FUNCTION DECISION")
print("=" * 70)
print(f"""
Results Summary:
  Cosine Similarity:   {metric_results['cosine']['accuracy']:.1%} accuracy, gap = {metric_results['cosine']['gap']:.4f}
  Euclidean (1/(1+d)): {metric_results['euclidean']['accuracy']:.1%} accuracy, gap = {metric_results['euclidean']['gap']:.4f}
  Manhattan (1/(1+d)): {metric_results['manhattan']['accuracy']:.1%} accuracy, gap = {metric_results['manhattan']['gap']:.4f}

Decision: USE EUCLIDEAN DISTANCE

Why:
  1. Highest pairwise accuracy — best at distinguishing liked from random songs
  2. Magnitude-sensitive — if a user likes high Energy (0.8), a song with
     Energy=0.8 scores higher than one with Energy=0.2
  3. Cosine fails here because all features are non-negative and similar scale,
     so most songs point in roughly the same "direction" (all scores > 0.89)
  4. Manhattan performs similarly to Euclidean but Euclidean is more standard
     in recommendation literature

Reward formula: reward = 1 / (1 + euclidean_distance)
  - Range: [0, 1]
  - Perfect match (distance=0) → reward = 1.0
  - Far mismatch (distance=2) → reward = 0.33
""")

---
## 2. Cluster Size (K) Test

**Goal:** Find the optimal number of K-means clusters for the bandit's arm space.

**Method:** For each K value (5, 10, 15, 20, 25, 30, 35, 40, 45, 50):
- Cluster all 19,675 songs using K-means on the 7 taste features
- Run a simple epsilon-greedy bandit simulation (5,000 steps)
- Measure average reward, cold-start performance, and learning improvement

**Trade-off:**
- Fewer clusters → easier to explore (fewer arms) but imprecise matching
- More clusters → precise matching but harder for the bandit to explore all arms

In [ ]:
K_VALUES = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
T = 5000
EPS = 0.1

k_results = {}
k_rewards_over_time = {}

print("=" * 90)
print(f"CLUSTER SIZE TEST (eps-greedy, eps={EPS}, T={T})")
print("=" * 90)
print()

user_ids = profiles.index.values
X = songs[FEATS].values

for K in K_VALUES:
    # Cluster songs
    km = KMeans(n_clusters=K, random_state=42, n_init=10)
    labels = km.fit_predict(X)

    # Precompute songs per cluster
    cluster_songs = {}
    for c in range(K):
        cluster_songs[c] = X[labels == c]

    # Run epsilon-greedy simulation
    np.random.seed(42)
    arm_rewards = np.zeros(K)
    arm_counts = np.zeros(K)
    rewards_over_time = []

    for t in range(T):
        pid = np.random.choice(user_ids)
        user_profile = profiles.loc[pid, FEATS].values

        # Epsilon-greedy
        if np.random.random() < EPS or arm_counts.sum() < K:
            arm = np.random.randint(K)
        else:
            arm = np.argmax(arm_rewards / np.maximum(arm_counts, 1))

        # Pick random song from cluster
        song = cluster_songs[arm][np.random.randint(len(cluster_songs[arm]))]

        # Reward
        dist = np.linalg.norm(song - user_profile)
        reward = 1 / (1 + dist)

        arm_rewards[arm] += reward
        arm_counts[arm] += 1
        rewards_over_time.append(reward)

    rewards = np.array(rewards_over_time)
    first_500 = rewards[:500].mean()
    last_500 = rewards[-500:].mean()

    k_results[K] = {
        'avg_reward': rewards.mean(),
        'cold_start': first_500,
        'converged': last_500,
        'improvement': last_500 - first_500,
        'avg_cluster_size': len(songs) // K,
        'min_cluster': min(len(v) for v in cluster_songs.values()),
        'max_cluster': max(len(v) for v in cluster_songs.values()),
    }
    k_rewards_over_time[K] = rewards

    print(f"K = {K:>3d}  |  Avg: {rewards.mean():.4f}  |  "
          f"Cold start: {first_500:.4f}  |  Converged: {last_500:.4f}  |  "
          f"Improvement: {last_500 - first_500:+.4f}  |  "
          f"Cluster size: {len(songs)//K} ({min(len(v) for v in cluster_songs.values())}-{max(len(v) for v in cluster_songs.values())})")

In [ ]:
# Visualize learning curves for each K
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Plot 1: Smoothed learning curves
window = 200
for K in K_VALUES:
    rewards = k_rewards_over_time[K]
    smoothed = pd.Series(rewards).rolling(window=window).mean()
    axes[0].plot(smoothed, label=f'K={K}', alpha=0.8)

axes[0].set_xlabel('Step')
axes[0].set_ylabel('Average Reward (rolling mean)')
axes[0].set_title(f'Learning Curves by Cluster Size (smoothed, window={window})')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Plot 2: Bar chart comparison
x = np.arange(len(K_VALUES))
width = 0.35
cold = [k_results[K]['cold_start'] for K in K_VALUES]
conv = [k_results[K]['converged'] for K in K_VALUES]

axes[1].bar(x - width/2, cold, width, label='Cold Start (first 500)', color='salmon')
axes[1].bar(x + width/2, conv, width, label='Converged (last 500)', color='seagreen')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Average Reward')
axes[1].set_title('Cold Start vs Converged Performance')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'K={K}' for K in K_VALUES], fontsize=8)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Plot 3: Improvement by K
fig2, ax = plt.subplots(figsize=(10, 5))
improvements = [k_results[K]['improvement'] for K in K_VALUES]
colors = ['seagreen' if imp > 0 else 'salmon' for imp in improvements]
ax.bar(range(len(K_VALUES)), improvements, color=colors)
ax.set_xticks(range(len(K_VALUES)))
ax.set_xticklabels([f'K={K}' for K in K_VALUES])
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Improvement (Converged - Cold Start)')
ax.set_title('Learning Improvement by Cluster Size')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 90)
print("CLUSTER SIZE DECISION")
print("=" * 90)
print()

print(f"{'K':>5s}  {'Avg Reward':>12s}  {'Cold Start':>12s}  {'Converged':>12s}  {'Improvement':>12s}")
print("-" * 65)
for K in K_VALUES:
    r = k_results[K]
    marker = " <--" if K == 20 else ""
    print(f"{K:>5d}  {r['avg_reward']:>12.4f}  {r['cold_start']:>12.4f}  {r['converged']:>12.4f}  {r['improvement']:>+12.4f}{marker}")

print(f"""
Decision: USE K=20 CLUSTERS

Full analysis across K=5 to K=50 (step 5):

  Key observations:
  - K=40 shows the largest raw learning improvement (+0.027), suggesting
    finer clusters can help the bandit learn sharper distinctions.
  - K=25 and K=30 show NEGATIVE improvement — the bandit performs worse
    over time at these values, likely due to an awkward balance between
    too many arms to explore and not enough precision gain.
  - K=50 has the highest average reward but flat learning curves.

  Why K=20 is still the best choice:
  1. Consistent positive learning — improvement is reliable (+0.012),
     whereas K=40's larger gain may be noisy (single simulation run)
  2. Manageable arm space — 20 arms is feasible for all bandit algorithms
     (epsilon-greedy, UCB, Thompson Sampling, LinUCB) to explore within
     reasonable step counts. K=40 doubles the exploration cost.
  3. Meaningful cluster sizes — ~980 songs per cluster gives stable,
     well-defined music "genres" vs ~490 for K=40
  4. K=5 and K=10 are too coarse — clusters contain 2000-4000 songs,
     so even the "right" cluster has many mismatched songs
  5. K=25-30 are unstable — negative improvement means the bandit
     cannot reliably learn at these values
  6. K=50 scores highest overall but learning is flat — less interesting
     for demonstrating that bandit algorithms actually adapt

  Note: K=40 could be included as a secondary analysis point to discuss
  the trade-off between exploration cost and learning precision.

Reward formula: reward = 1 / (1 + euclidean_distance)
Recommended K: 20
""")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# §3-§4 SETUP — contextual-bandit sensitivity sweeps
# ═══════════════════════════════════════════════════════════════════
# Fix K=20 (decided in §2). Rebuild cluster_songs at that K. Define the
# two contextual algorithms whose hyper-parameters we are tuning. These
# class definitions mirror 05 exactly — 05 will import the same values
# we decide here.

K = 20
d = len(FEATS)          # 7

km_final = KMeans(n_clusters=K, random_state=42, n_init=10)
song_labels = km_final.fit_predict(X)
cluster_songs = {c: X[song_labels == c] for c in range(K)}
profiles_arr = profiles[FEATS].values
n_users = len(profiles_arr)


class LinUCB:
    """Contextual. Disjoint linear model per arm + UCB exploration.
    Reference: Li et al. (2010)"""
    def __init__(self, K, d, alpha=0.5):
        self.K = K
        self.d_ctx = d + 1
        self.alpha = alpha
        self.A_inv = [np.eye(self.d_ctx) for _ in range(K)]
        self.b = [np.zeros(self.d_ctx) for _ in range(K)]
    def select(self, context):
        x = np.concatenate([[1.0], context])
        best_arm, best_val = 0, -np.inf
        for a in range(self.K):
            theta = self.A_inv[a] @ self.b[a]
            pred = x @ theta
            bonus = self.alpha * np.sqrt(x @ self.A_inv[a] @ x)
            val = pred + bonus
            if val > best_val:
                best_val, best_arm = val, a
        return best_arm
    def update(self, arm, reward, context):
        x = np.concatenate([[1.0], context])
        Ainv_x = self.A_inv[arm] @ x
        denom = 1.0 + x @ Ainv_x
        self.A_inv[arm] -= np.outer(Ainv_x, Ainv_x) / denom
        self.b[arm] += reward * x


class ContextualTS:
    """Contextual Thompson Sampling. Bayesian linear model per arm.
    Reference: Agrawal & Goyal (2013)"""
    def __init__(self, K, d, nu=0.3):
        self.K = K
        self.d_ctx = d + 1
        self.nu = nu
        self.A_inv = [np.eye(self.d_ctx) for _ in range(K)]
        self.b = [np.zeros(self.d_ctx) for _ in range(K)]
    def select(self, context):
        x = np.concatenate([[1.0], context])
        best_arm, best_val = 0, -np.inf
        for a in range(self.K):
            mu = self.A_inv[a] @ self.b[a]
            theta_sample = np.random.multivariate_normal(
                mu, self.nu**2 * self.A_inv[a])
            val = x @ theta_sample
            if val > best_val:
                best_val, best_arm = val, a
        return best_arm
    def update(self, arm, reward, context):
        x = np.concatenate([[1.0], context])
        Ainv_x = self.A_inv[arm] @ x
        denom = 1.0 + x @ Ainv_x
        self.A_inv[arm] -= np.outer(Ainv_x, Ainv_x) / denom
        self.b[arm] += reward * x


# Sensitivity-sweep scale — matches the main 05 experiment so the numbers
# transfer verbatim. This notebook is heavy but runs once.
N_SIMS = 10
T = 50_000

print(f"Setup: K={K}, d={d}, N_SIMS={N_SIMS}, T={T:,}, users={n_users}")
print(f"Cluster sizes: min={min(len(c) for c in cluster_songs.values())}, "
      f"max={max(len(c) for c in cluster_songs.values())}")


---
## 3. LinUCB α Sensitivity

**Goal:** Choose the exploration weight α for LinUCB's UCB bonus  
\(a^* = \arg\max_a \left[ \mathbf{x}^T\hat{\boldsymbol\theta}_a + \alpha\sqrt{\mathbf{x}^T\mathbf{A}_a^{-1}\mathbf{x}} \right]\).

**Method:** Sweep α ∈ {0.3, 0.5, 0.8, 1.0, 1.5} at the full main-experiment scale (N=10 sims, T=50,000 steps). Same reward function and K as 05.

**Trade-off:** small α → faster convergence but risks locking onto a sub-optimal arm; large α → more exploration, slower convergence. Compare cold-start (first 500) vs. converged (last 500) means.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# §3. LINUCB ALPHA SENSITIVITY
# ═══════════════════════════════════════════════════════════════════
print('=' * 75)
print('LINUCB ALPHA SENSITIVITY')
print('=' * 75)

alpha_values = [0.3, 0.5, 0.8, 1.0, 1.5]
alpha_results = {a: np.zeros((N_SIMS, T)) for a in alpha_values}

for sim in range(N_SIMS):
    user_rng = np.random.default_rng(sim * 42 + 7)
    user_seq = user_rng.integers(0, n_users, size=T)

    for alpha in alpha_values:
        np.random.seed(sim * 1000 + int(alpha * 100))
        algo = LinUCB(K, d, alpha=alpha)
        for t in range(T):
            ctx = profiles_arr[user_seq[t]]
            arm = algo.select(ctx)
            song = cluster_songs[arm][np.random.randint(len(cluster_songs[arm]))]
            dist = np.linalg.norm(song - ctx)
            reward = 1.0 / (1.0 + dist)
            algo.update(arm, reward, ctx)
            alpha_results[alpha][sim, t] = reward

    print(f'  Alpha sweep simulation {sim + 1}/{N_SIMS} complete')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
window = 200
for alpha in alpha_values:
    smoothed_sims = np.zeros((N_SIMS, T))
    for s in range(N_SIMS):
        smoothed_sims[s] = pd.Series(alpha_results[alpha][s]).rolling(window=window).mean().values
    mean_curve = np.nanmean(smoothed_sims, axis=0)
    std_curve = np.nanstd(smoothed_sims, axis=0)
    axes[0].plot(mean_curve, label=f'α={alpha}', linewidth=2.5, alpha=0.85)
    axes[0].fill_between(range(T), mean_curve - std_curve, mean_curve + std_curve, alpha=0.1)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Average Reward')
axes[0].set_title('LinUCB Learning Curves by α')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

cold_vals = [alpha_results[a][:, :500].mean() for a in alpha_values]
conv_vals = [alpha_results[a][:, -500:].mean() for a in alpha_values]
x_pos = np.arange(len(alpha_values)); width = 0.35
axes[1].bar(x_pos - width/2, cold_vals, width, label='Cold Start (first 500)', color='salmon', alpha=0.8)
axes[1].bar(x_pos + width/2, conv_vals, width, label='Converged (last 500)', color='seagreen', alpha=0.8)
axes[1].set_xticks(x_pos); axes[1].set_xticklabels([f'α={a}' for a in alpha_values])
axes[1].set_ylabel('Average Reward')
axes[1].set_title('Cold Start vs Converged')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

print('\nAlpha Value Comparison:')
print('─' * 70)
print(f'{"Alpha":<8s}  {"Cold Start":>12s}  {"Converged":>12s}  {"Improvement":>12s}')
print('─' * 70)
for alpha in alpha_values:
    cold = alpha_results[alpha][:, :500].mean()
    conv = alpha_results[alpha][:, -500:].mean()
    mark = ' ← (chosen)' if alpha == 0.5 else ''
    print(f'{alpha:<8.1f}  {cold:>12.4f}  {conv:>12.4f}  {conv-cold:>+12.4f}{mark}')
print()
print('Decision: α = 0.5 — best converged reward with competitive cold-start.')


---
## 4. Contextual TS ν Sensitivity

**Goal:** Choose the posterior-sampling scale ν for Contextual TS  
\(\tilde{\boldsymbol\theta}_a \sim \mathcal{N}(\hat{\boldsymbol\mu}_a,\,\nu^2\mathbf{A}_a^{-1})\).

**Method:** Sweep ν ∈ {0.1, 0.3, 0.5, 0.8, 1.0} at the full main-experiment scale (N=10 sims, T=50,000 steps).

**Trade-off:** small ν → tight posterior, fast convergence but may miss good arms; large ν → wide posterior, more exploration.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# §4. CONTEXTUAL TS NU SENSITIVITY
# ═══════════════════════════════════════════════════════════════════
print('=' * 75)
print('CONTEXTUAL TS NU SENSITIVITY')
print('=' * 75)

nu_values = [0.1, 0.3, 0.5, 0.8, 1.0]
nu_results = {n: np.zeros((N_SIMS, T)) for n in nu_values}

for sim in range(N_SIMS):
    user_rng = np.random.default_rng(sim * 42 + 7)
    user_seq = user_rng.integers(0, n_users, size=T)

    for nu in nu_values:
        np.random.seed(sim * 1000 + int(nu * 100))
        algo = ContextualTS(K, d, nu=nu)
        for t in range(T):
            ctx = profiles_arr[user_seq[t]]
            arm = algo.select(ctx)
            song = cluster_songs[arm][np.random.randint(len(cluster_songs[arm]))]
            dist = np.linalg.norm(song - ctx)
            reward = 1.0 / (1.0 + dist)
            algo.update(arm, reward, ctx)
            nu_results[nu][sim, t] = reward

    print(f'  Nu sweep simulation {sim + 1}/{N_SIMS} complete')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
window = 200
for nu in nu_values:
    smoothed_sims = np.zeros((N_SIMS, T))
    for s in range(N_SIMS):
        smoothed_sims[s] = pd.Series(nu_results[nu][s]).rolling(window=window).mean().values
    mean_curve = np.nanmean(smoothed_sims, axis=0)
    std_curve = np.nanstd(smoothed_sims, axis=0)
    axes[0].plot(mean_curve, label=f'ν={nu}', linewidth=2.5, alpha=0.85)
    axes[0].fill_between(range(T), mean_curve - std_curve, mean_curve + std_curve, alpha=0.1)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Average Reward')
axes[0].set_title('Contextual TS Learning Curves by ν')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

cold_vals = [nu_results[n][:, :500].mean() for n in nu_values]
conv_vals = [nu_results[n][:, -500:].mean() for n in nu_values]
x_pos = np.arange(len(nu_values)); width = 0.35
axes[1].bar(x_pos - width/2, cold_vals, width, label='Cold Start (first 500)', color='salmon', alpha=0.8)
axes[1].bar(x_pos + width/2, conv_vals, width, label='Converged (last 500)', color='seagreen', alpha=0.8)
axes[1].set_xticks(x_pos); axes[1].set_xticklabels([f'ν={n}' for n in nu_values])
axes[1].set_ylabel('Average Reward')
axes[1].set_title('Cold Start vs Converged')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

print('\nNu Value Comparison:')
print('─' * 70)
print(f'{"Nu":<8s}  {"Cold Start":>12s}  {"Converged":>12s}  {"Improvement":>12s}')
print('─' * 70)
for nu in nu_values:
    cold = nu_results[nu][:, :500].mean()
    conv = nu_results[nu][:, -500:].mean()
    mark = ' ← (chosen)' if nu == 0.3 else ''
    print(f'{nu:<8.1f}  {cold:>12.4f}  {conv:>12.4f}  {conv-cold:>+12.4f}{mark}')
print()
print('Decision: ν = 0.3 — wins on both cold-start and converged reward.')


---
## 5. Non-Contextual Bandit Parameters

The contextual sweeps above (§3, §4) cover α and ν, but four other hyper-parameters are hardcoded in 05's algorithm registry. This section sweeps them at the same N=10, T=50,000 scale.

- **ε** for `EpsilonGreedy` — exploration rate
- **c** for `UCB1` — exploration-bonus coefficient
- **B** for `BootstrapTS` — number of online bootstrap replicates
- **γ** for `DecayingEpsilonGreedy_Exponential` — multiplicative decay per step

Non-contextual algorithms run fast (no matrix ops), so all four sweeps together finish in ~1 min.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# §5 SETUP — non-contextual algorithm definitions
# ═══════════════════════════════════════════════════════════════════
# These mirror 05 exactly. No context used — `select()` ignores user.

class EpsilonGreedy_NC:
    def __init__(self, K, eps=0.1):
        self.K, self.eps = K, eps
        self.counts = np.zeros(K); self.values = np.zeros(K)
    def select(self):
        if np.random.random() < self.eps or np.min(self.counts) == 0:
            return np.random.randint(self.K)
        return int(np.argmax(self.values))
    def update(self, a, r):
        self.counts[a] += 1
        self.values[a] += (r - self.values[a]) / self.counts[a]


class UCB1_NC:
    def __init__(self, K, c=2.0):
        self.K, self.c = K, c
        self.counts = np.zeros(K); self.values = np.zeros(K); self.total = 0
    def select(self):
        unplayed = np.where(self.counts == 0)[0]
        if len(unplayed) > 0: return int(unplayed[0])
        ucb = self.values + self.c * np.sqrt(np.log(self.total) / self.counts)
        return int(np.argmax(ucb))
    def update(self, a, r):
        self.counts[a] += 1; self.total += 1
        self.values[a] += (r - self.values[a]) / self.counts[a]


class BootstrapTS_NC:
    def __init__(self, K, B=100):
        self.K, self.B = K, B
        self.counts = np.zeros(K)
        self.sum_w  = np.zeros((K, B))
        self.sum_wr = np.zeros((K, B))
    def select(self):
        unplayed = np.where(self.counts == 0)[0]
        if len(unplayed) > 0: return int(unplayed[0])
        b_idx = np.random.randint(self.B, size=self.K)
        sw  = self.sum_w[np.arange(self.K),  b_idx]
        swr = self.sum_wr[np.arange(self.K), b_idx]
        means = np.where(sw > 0, swr / np.maximum(sw, 1e-12), 0.0)
        return int(np.argmax(means))
    def update(self, a, r):
        self.counts[a] += 1
        w = np.random.poisson(1.0, size=self.B).astype(float)
        self.sum_w[a]  += w
        self.sum_wr[a] += w * r


class DecayingEG_Exp_NC:
    def __init__(self, K, eps=0.1, decay=0.9999):
        self.K, self.eps, self.decay = K, eps, decay
        self.counts = np.zeros(K); self.values = np.zeros(K); self.eps_t = eps
    def select(self):
        if np.random.random() < self.eps_t or np.min(self.counts) == 0:
            return np.random.randint(self.K)
        return int(np.argmax(self.values))
    def update(self, a, r):
        self.counts[a] += 1
        self.values[a] += (r - self.values[a]) / self.counts[a]
        self.eps_t *= self.decay


def run_nc_sweep(ctor, param_name, values, seed_mul=100):
    """Generic sweep runner for non-contextual algorithms."""
    res = {v: np.zeros((N_SIMS, T)) for v in values}
    for sim in range(N_SIMS):
        user_rng = np.random.default_rng(sim * 42 + 7)
        user_seq = user_rng.integers(0, n_users, size=T)
        for v in values:
            np.random.seed(sim * 1000 + int(v * seed_mul) % 10000)
            algo = ctor(v)
            for t in range(T):
                ctx = profiles_arr[user_seq[t]]
                arm = algo.select()
                song = cluster_songs[arm][np.random.randint(len(cluster_songs[arm]))]
                dist = np.linalg.norm(song - ctx)
                reward = 1.0 / (1.0 + dist)
                algo.update(arm, reward)
                res[v][sim, t] = reward
        print(f'  sim {sim+1}/{N_SIMS} done')
    return res


def summarize_sweep(res, values, param_label, chosen):
    print(f'\n{param_label:<10s}  {"Cold Start":>12s}  {"Converged":>12s}  {"Improvement":>12s}  {"σ(conv)":>10s}')
    print('─' * 72)
    for v in values:
        cold = res[v][:, :500].mean()
        conv = res[v][:, -500:].mean()
        sd   = res[v][:, -500:].mean(axis=1).std(ddof=1)
        mark = ' ← (chosen)' if v == chosen else ''
        print(f'{str(v):<10s}  {cold:>12.4f}  {conv:>12.4f}  {conv-cold:>+12.4f}  {sd:>10.4f}{mark}')


def plot_sweep(res, values, param_label, title):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    window = 200
    for v in values:
        sm = np.zeros((N_SIMS, T))
        for s in range(N_SIMS):
            sm[s] = pd.Series(res[v][s]).rolling(window=window).mean().values
        m = np.nanmean(sm, axis=0); sd = np.nanstd(sm, axis=0)
        axes[0].plot(m, label=f'{param_label}={v}', linewidth=2, alpha=0.85)
        axes[0].fill_between(range(T), m - sd, m + sd, alpha=0.1)
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Average Reward')
    axes[0].set_title(f'{title} — Learning Curves')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    cold = [res[v][:, :500].mean() for v in values]
    conv = [res[v][:, -500:].mean() for v in values]
    x_pos = np.arange(len(values)); w = 0.35
    axes[1].bar(x_pos - w/2, cold, w, label='Cold Start', color='salmon', alpha=0.8)
    axes[1].bar(x_pos + w/2, conv, w, label='Converged',  color='seagreen', alpha=0.8)
    axes[1].set_xticks(x_pos); axes[1].set_xticklabels([f'{param_label}={v}' for v in values])
    axes[1].set_ylabel('Average Reward'); axes[1].set_title(f'{title} — Cold vs Converged')
    axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')
    plt.tight_layout(); plt.show()


print('§5 setup complete: 4 non-contextual algorithm classes and sweep helpers defined.')


### 5.1 ε for EpsilonGreedy

**Current default:** ε=0.1 (textbook).  
**Sweep:** {0.01, 0.05, 0.1, 0.2, 0.3}.  
**Expected:** rewards are dense in [0,1] so tiny exploration should suffice.


In [ ]:
print('=' * 75); print('EPSILON-GREEDY ε SWEEP'); print('=' * 75)
eps_values = [0.01, 0.05, 0.1, 0.2, 0.3]
eps_res = run_nc_sweep(lambda v: EpsilonGreedy_NC(K, eps=v), 'ε', eps_values)
plot_sweep(eps_res, eps_values, 'ε', 'EpsilonGreedy')
summarize_sweep(eps_res, eps_values, 'ε', chosen=0.01)
print('\nDecision: ε = 0.01 — converged reward ~3σ above ε=0.1 default.')


### 5.2 c for UCB1

**Current default:** c=2.0 (Sutton-Barto textbook).  
**Sweep:** {0.5, 1.0, 2.0, 4.0}.  
**Expected:** since rewards ∈ [0,1] (not [0,100]), the standard c=2 may over-explore. A smaller c should win.


In [ ]:
print('=' * 75); print('UCB1 c SWEEP'); print('=' * 75)
c_values = [0.5, 1.0, 2.0, 4.0]
c_res = run_nc_sweep(lambda v: UCB1_NC(K, c=v), 'c', c_values)
plot_sweep(c_res, c_values, 'c', 'UCB1')
summarize_sweep(c_res, c_values, 'c', chosen=0.5)
print('\nDecision: c = 0.5 — strong ~10σ win over c=2 default on bounded rewards.')


### 5.3 B for Bootstrap Thompson Sampling

**Current default:** B=100.  
**Sweep:** {20, 50, 100, 200}.  
**Expected:** bootstrap variance shrinks with B but plateaus early; B should be insensitive above ~20.


In [ ]:
print('=' * 75); print('BOOTSTRAP TS B SWEEP'); print('=' * 75)
b_values = [20, 50, 100, 200]
b_res = run_nc_sweep(lambda v: BootstrapTS_NC(K, B=int(v)), 'B', b_values)
plot_sweep(b_res, b_values, 'B', 'Bootstrap TS')
summarize_sweep(b_res, b_values, 'B', chosen=100)
print('\nDecision: B = 100 — all values within 1σ; keep default.')


### 5.4 γ for Exponential-Decay ε-Greedy

**Current default in code:** γ=0.9999.  
**Sweep:** {0.999, 0.9995, 0.9999, 0.99995}.  
**Expected:** slower decay (γ closer to 1) keeps exploration alive longer. For T=50,000 the effective horizon matters: γ=0.999 decays ε to ~0 by t≈7,000 (too fast); γ=0.9999 sustains exploration to ~70,000.


In [ ]:
print('=' * 75); print('EXPONENTIAL EG γ SWEEP'); print('=' * 75)
gamma_values = [0.999, 0.9995, 0.9999, 0.99995]
g_res = run_nc_sweep(lambda v: DecayingEG_Exp_NC(K, eps=0.1, decay=v), 'γ', gamma_values)
plot_sweep(g_res, gamma_values, 'γ', 'Exponential-Decay ε-Greedy')
summarize_sweep(g_res, gamma_values, 'γ', chosen=0.9999)
print('\nDecision: γ = 0.9999 — highest converged reward with lowest variance.')
print('Note: report text cites γ=0.999, which this sweep shows is inferior.')


---
## Final Hyper-parameter Decisions

| Algorithm | Param | Decision | Sweep source |
|---|---|---|---|
| Reward function | — | euclidean `1/(1+d)` | §1 |
| K-means | K | 20 | §2 |
| LinUCB | α | 0.5 | §3 |
| Contextual TS | ν | 0.3 | §4 |
| EpsilonGreedy | ε | 0.01 | §5.1 |
| UCB1 | c | 0.5 | §5.2 |
| BootstrapTS | B | 100 | §5.3 |
| Exp-decay ε-Greedy | γ | 0.9999 | §5.4 |

These values are consumed verbatim by `05_bandit_experiments.ipynb`.


---
## 6. Robustness Checks

The sweeps in §3–§5 each print a "chosen" value based on highest converged mean. Two caveats were deliberately deferred and are addressed here:

1. **Statistical significance of the ranking** — *is the best value actually distinguishable from its runner-up?* With N=10 Monte Carlo replications using a **paired design** (same user sequence across parameter values within each sim), a paired t-test on per-simulation converged means is the right tool. We also report 95% CIs on each mean.

2. **Window-size sensitivity of "converged"** — *we defined "converged" as the last 500 steps, which is arbitrary.* If the ranking flips at W=1000 or W=2000, the decision was window-dependent. We re-rank every sweep across W ∈ {200, 500, 1000, 2000}.


### 6.1 Paired t-tests and 95% Confidence Intervals

For each sweep, we compute the per-simulation mean over the last 500 steps (yielding N=10 values per parameter), then:

- **95% CI** on each value's mean: $\bar{x} \pm 1.96 \cdot \text{SE}$ where $\text{SE} = s / \sqrt{N}$.
- **Paired t-test** between the winner and every other value. Paired because within each simulation all values see identical user arrivals and song draws — this removes simulation-level variance.
- Significance flags: `*` p<0.05, `**` p<0.01, `***` p<0.001. N=10 is small, so marginal p-values should be interpreted cautiously.


In [ ]:
from scipy import stats

def stat_report(res, values, param_label, chosen, window=500):
    """Paired-design statistical summary for a sweep result dict.
    
    res: {value: np.ndarray(N_SIMS, T)}
    Returns printed table; does not modify state.
    """
    per_sim = {v: res[v][:, -window:].mean(axis=1) for v in values}
    winner = per_sim[chosen]
    N = len(winner)

    print(f'\n{param_label:<10s}  {"Mean":>8s}  {"95% CI":>22s}  {"Δ vs winner":>13s}  {"paired-t p":>12s}')
    print('─' * 80)
    for v in values:
        x = per_sim[v]
        m = x.mean()
        se = x.std(ddof=1) / np.sqrt(N) if N > 1 else 0.0
        ci_lo, ci_hi = m - 1.96*se, m + 1.96*se
        if v == chosen:
            pstr = '— (winner)'
            dstr = '  baseline'
        else:
            t, p = stats.ttest_rel(winner, x)
            flag = ' ***' if p < 0.001 else (' **' if p < 0.01 else (' *' if p < 0.05 else ''))
            pstr = f'{p:.4f}{flag}'
            dstr = f'{per_sim[chosen].mean() - m:+.4f}'
        print(f'{str(v):<10s}  {m:>8.4f}  [{ci_lo:.4f}, {ci_hi:.4f}]  {dstr:>13s}  {pstr:>12s}')


print('=' * 80); print('§6.1  PAIRED t-TESTS — is the ranking statistically significant?'); print('=' * 80)

stat_report(alpha_results, alpha_values, 'α (LinUCB)',         chosen=0.5)
stat_report(nu_results,    nu_values,    'ν (CtxTS)',          chosen=0.3)
stat_report(eps_res,       eps_values,   'ε (EpsGreedy)',      chosen=0.01)
stat_report(c_res,         c_values,     'c (UCB1)',           chosen=0.5)
stat_report(b_res,         b_values,     'B (BootstrapTS)',    chosen=100)
stat_report(g_res,         gamma_values, 'γ (ExpDecayEG)',     chosen=0.9999)

print()
print('Reading this table:')
print('  • p < 0.05 (*) → the winner is reliably better than this competitor.')
print('  • p ≥ 0.05    → the two values are indistinguishable at N=10. Decision is')
print('                  then a tie — fall back to the lower-variance or lower-risk choice.')
print('  • Bootstrap TS B values are expected to be non-significant (all within 1σ).')


### 6.2 Convergence-Window Sensitivity

We defined "converged" as the last 500 steps. To check this isn't load-bearing, we re-rank each sweep across W ∈ {200, 500, 1000, 2000}. A decision is robust if the winner is the same across all windows (or the top two values only swap within their CI).


In [ ]:
print('=' * 80)
print('§6.2  WINDOW SENSITIVITY — does the winner depend on window W?')
print('=' * 80)

all_sweeps = [
    ('α (LinUCB)',      alpha_results, alpha_values, 0.5),
    ('ν (CtxTS)',       nu_results,    nu_values,    0.3),
    ('ε (EpsGreedy)',   eps_res,       eps_values,   0.01),
    ('c (UCB1)',        c_res,         c_values,     0.5),
    ('B (BootstrapTS)', b_res,         b_values,     100),
    ('γ (ExpDecayEG)',  g_res,         gamma_values, 0.9999),
]
windows = [200, 500, 1000, 2000]

header = f'{"Sweep":<18s}  {"chosen":>8s}  '
for w in windows:
    header += f'{"W="+str(w):>12s}  '
print()
print(header)
print('─' * len(header))

for label, res, values, chosen in all_sweeps:
    row = f'{label:<18s}  {str(chosen):>8s}  '
    for w in windows:
        means = {v: res[v][:, -w:].mean() for v in values}
        best = max(means, key=means.get)
        mark = '✓' if best == chosen else '✗'
        cell = f'{str(best)} {mark}'
        row += f'{cell:>12s}  '
    print(row)

print()
print('Reading this table:')
print('  • ✓ = winner at this W matches the §3–§5 chosen value — decision is window-robust.')
print('  • ✗ = winner flips — re-examine the decision.')
print('  • B (BootstrapTS): flips are expected since all values sit within 1σ of each other.')
